# M5 Benchmarks v2 — Full Method Suite

Runs all 12 R benchmark methods with minimal memory footprint.
Each method is run, evaluated, and freed before the next begins —
no more than one forecast DataFrame is held in memory at a time.

Methods: `Naive`, `sNaive`, `SES`, `MA`, `Croston`, `optCroston`, `SBA`, `TSB`, `ADIDA`, `iMAPA`, `ES_bu`, `ARIMA_bu`

In [1]:
import sys
sys.path.append("/home/nmwamsojo/tsfm-explo/src/jobs/")

import os
import gc
import ctypes

import pandas as pd
import torch
from IPython.display import display

from m5_benchmarks import M5BenchmarkSuite
from m5_dataprep import M5DataPipeline
from m5_evaluator import M5Evaluator

# Config

In [ ]:
DATA_PATH     = "/mnt/lab/datasets/M5/jointed_M5.parquet"
CALENDAR_PATH = "/mnt/lab/nmwamsojo/m5_data/calendar.csv"
ACTUALS_PATH  = "/mnt/lab/nmwamsojo/m5_data/sales_test_evaluation.csv"

DATA_TAG   = "default"
FORCE_REFIT = False   # True → re-run even if a cached forecast exists

CUTOFF_DAY = (
    pd.to_datetime("2016-05-22") - pd.Timedelta(days=0)
).strftime("%Y-%m-%d")
print(f"Cutoff day : {CUTOFF_DAY}")

# Methods to run — remove any you don't need
ALL_METHODS = [
    "Naive", "sNaive", "SES", "MA",
    "Croston", "optCroston", "SBA", "TSB",
    "ADIDA", "iMAPA",
    "ES_bu", "ARIMA_bu",
]

ALL_METHODS = [
    "ES_bu"
]

METHODS_TO_RUN = ALL_METHODS

wrapper_dict = {
    "cutoff_day":          CUTOFF_DAY,
    "level":               12,
    "data_tag":            DATA_TAG,
    "target":              "sales_quantity",
    "eval_metric":         "RMSSE",
    "enable_ensemble":     False,
    "skip_model_selection": True,
    "verbosity":            0,
}
print(f"Methods to run : {METHODS_TO_RUN}")
print(f"Force refit    : {FORCE_REFIT}")

Cutoff day : 2016-04-24
Methods to run : ['ES_bu']
Force refit    : True


# Data

In [3]:
pipeline = M5DataPipeline(config={"tag": DATA_TAG})
hist_df, hist_df_trimmed, future_df, static_df, weights_scales = (
    pipeline.get_prepared_data(DATA_PATH, CUTOFF_DAY, level=12, force_reprepare=False)
)

print(f"hist_df         : {hist_df.shape}")
print(f"hist_df_trimmed : {hist_df_trimmed.shape}")
print(f"future_df       : {future_df.shape}")
print(f"static_df       : {static_df.shape}")

del pipeline
gc.collect()

--- Cache Hit: Data found in /mnt/lab/nmwamsojo/prepared_data/default/level_12/20160424 ---
hist_df         : (58327370, 21)
hist_df_trimmed : (45942500, 21)
future_df       : (853720, 20)
static_df       : (30490, 6)


0

In [9]:
def _wide_to_long(gt_wide: pd.DataFrame, calendar_path: str) -> pd.DataFrame:
    if "id" not in gt_wide.columns:
        gt_wide["id"] = gt_wide["item_id"] + "_" + gt_wide["store_id"] + "_evaluation"
    day_cols = [c for c in gt_wide.columns if c.startswith("d_")]
    long = gt_wide.melt(id_vars=["id"], value_vars=day_cols,
                        var_name="d", value_name="sales_quantity")
    cal = pd.read_csv(calendar_path, usecols=["d", "date"])
    cal["date"] = pd.to_datetime(cal["date"])
    long = long.merge(cal, on="d", how="left").drop(columns=["d"])
    long["id"] = long["id"].str.replace("_evaluation", "", regex=False)
    return long[["id", "date", "sales_quantity"]]

DATA_PATH     = "/mnt/lab/datasets/M5/jointed_M5.parquet"
CALENDAR_PATH = "/mnt/lab/nmwamsojo/m5_data/calendar.csv"
ACTUALS_PATH  = "/mnt/lab/nmwamsojo/m5_data/sales_test_evaluation.csv"

DATA_TAG = "sales_only"

if CUTOFF_DAY == "2016-05-22":
    _eval_raw = pd.read_csv(ACTUALS_PATH)
    df_actual = _wide_to_long(_eval_raw, CALENDAR_PATH)
    del _eval_raw
else:
    df_actual = (
        pd.read_parquet(DATA_PATH, columns=["id", "date", "sold"])
        .rename(columns={"sold": "sales_quantity"})
    )
    df_actual["id"] = (
        df_actual["id"].astype(str)
        .str.replace("_evaluation", "", regex=False)
        .str.replace("_validation",  "", regex=False)
    )

print(f"Actuals : {df_actual.shape}  |  "
      f"{df_actual['date'].min().date()} \u2192 {df_actual['date'].max().date()}")

Actuals : (59181090, 3)  |  2011-01-29 → 2016-05-22


# Benchmarks — one method at a time

In [5]:
evaluator = M5Evaluator(
    raw_train_df     = hist_df,
    trimmed_train_df = hist_df_trimmed,
    static_df        = static_df,
    weights_df       = weights_scales,
    target_col       = "sales_quantity",
    price_col        = "sell_price",
)

suite = M5BenchmarkSuite(
    horizon    = 28,
    model_path = "/mnt/lab/nmwamsojo/autogluon_models/benchmarks/",
    base_dir   = "/mnt/lab/nmwamsojo/prepared_data/",
    n_jobs     = -1,
)
print("Evaluator and suite ready.")

  [CPU] CuPy not available — scale computation on 18 CPU cores.
  Building hierarchy scales and weights …
Evaluator and suite ready.


In [10]:
results_summary = []

for method in METHODS_TO_RUN:
    print(f"\n{'='*60}\nMethod: {method}\n{'='*60}")
    try:
        forecasts = suite.run(
            train_df     = hist_df_trimmed,
            methods      = [method],
            static_df    = static_df,
            wrapper_dict = wrapper_dict,
            force_refit  = FORCE_REFIT,
        )
        fcst = forecasts.pop(method)
        del forecasts

        metrics = evaluator.evaluate_all(fcst, df_actual)
        wrmsse  = float(metrics["WRMSSE"])
        wape    = float(metrics.get("WAPE_L12", float("nan")))
        results_summary.append({"method": method, "WRMSSE": wrmsse, "WAPE_L12": wape})
        print(f"  WRMSSE = {wrmsse:.4f}  |  WAPE = {wape:.2%}")

        del fcst, metrics

    except Exception as exc:
        print(f"  [FAILED] {exc}")
        results_summary.append({"method": method, "WRMSSE": float("nan"), "WAPE_L12": float("nan")})

    finally:
        gc.collect(2)
        try:
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except Exception:
            pass

print(f"\nDone — {len(results_summary)} methods processed.")


Method: ES_bu
--- Running ES_bu via AutoGluon (statsforecast backend) ---


Renaming existing column 'item_id' -> '__item_id' to avoid name collisions.


  WRMSSE = 0.7570  |  WAPE = 72.00%

Done — 1 methods processed.


# Results

In [ ]:
df_results = pd.DataFrame(results_summary).sort_values("WRMSSE").reset_index(drop=True)
display(
    df_results.style
    .format({"WRMSSE": "{:.4f}", "WAPE_L12": "{:.2%}"})
    .background_gradient(subset=["WRMSSE"], cmap="RdYlGn_r")
    .set_caption("Benchmark results — all methods, sorted by WRMSSE ↑ lower is better")
)
print(df_results.to_string(index=False))

,method,WRMSSE,WAPE_L12
0,ES_bu,nan,nan%


method  WRMSSE  WAPE_L12
 ES_bu     NaN       NaN
